# 🧭 Guía Metodológica: Proyectos de IA Generativa
## La Receta de Cocina para construir pipelines ML + GenAI

**Autor:** Guía didáctica basada en los notebooks del módulo IA Generativa  
**Nivel:** Intermedio — se asume experiencia en ML/DL clásico  
**Propósito:** Documento de referencia que seguirás en cada proyecto GenAI

---

### ¿Para qué sirve esta guía?

Si ya sabes hacer Machine Learning tradicional (clasificación, regresión, clustering), 
la IA Generativa añade un ingrediente nuevo: **modelos de lenguaje (LLMs)** que entienden 
y generan texto con una calidad comparable a la humana.

Esta guía te da el **proceso paso a paso** para combinar lo que ya sabes con las nuevas 
capacidades GenAI. Como una receta de cocina: si sigues los pasos en orden, el proyecto saldrá bien.

---

### Los 6 Patrones Fundamentales que aprenderás

| Patrón | Qué hace | Notebook de ejemplo |
|--------|----------|---------------------|
| **ML + Explicación** | El modelo predice, el LLM explica | T4.1 |
| **Extracción Estructurada** | Texto libre → JSON → ML downstream | T4.2 |
| **Feature Engineering semántico** | Texto → features numéricas para ML | T4.3 |
| **Agente con Herramientas** | LLM orquesta funciones ML/estadísticas | T4.4 |
| **Moderación en Cascada** | Reglas → ML → LLM (coste optimizado) | T4.5 |
| **LLM-as-Judge** | LLM evalúa la calidad de otro LLM | T4.6 |

# 🍳 PASO 1 — Define el Problema y Elige el Patrón Arquitectónico

## La pregunta más importante antes de escribir una línea de código

Antes de abrir un notebook, necesitas responder estas preguntas en orden:

---

### 1.1 ¿Qué tipo de dato de entrada tienes?

```
¿Texto libre? (emails, reseñas, tickets, documentos)
    → Los LLMs son tu herramienta principal para procesarlo

¿Datos tabulares + texto?  
    → Combinarás ML clásico + LLM (patrón de Feature Engineering)

¿Solo datos numéricos/tabulares?
    → Quizás solo necesites ML clásico + LLM para explicar resultados
```

### 1.2 ¿Qué quieres como salida?

```
¿Una predicción + explicación en lenguaje natural?
    → Patrón: ML → LLM Explicador (T4.1)

¿Datos estructurados extraídos de texto?
    → Patrón: LLM Extractor → JSON → ML (T4.2)

¿Features nuevas para mejorar un modelo?
    → Patrón: LLM Feature Engineer → ML enriquecido (T4.3)

¿Una interfaz conversacional sobre datos?
    → Patrón: Agente con Herramientas (T4.4)

¿Clasificación de contenido a escala?
    → Patrón: Cascada Reglas → ML → LLM (T4.5)

¿Evaluar calidad de respuestas LLM?
    → Patrón: LLM-as-Judge (T4.6)
```

### 1.3 ¿Cuáles son las restricciones del proyecto?

| Restricción | Impacto en diseño |
|-------------|-------------------|
| **Presupuesto API bajo** | Usa cascada: reglas → ML primero, LLM solo para casos difíciles |
| **Latencia < 200ms** | El LLM es lento (~1-3s), úsalo en batch o async |
| **Datos sensibles** | Modelos locales (Llama, Mistral) en lugar de APIs cloud |
| **Escala masiva** | Procesamiento en batch, no en tiempo real |
| **Interpretabilidad requerida** | LLM explicador obligatorio |

---

> **Regla de oro:** Empieza con el modelo más simple posible. Añade complejidad solo si los resultados no son suficientes.

# 🔧 PASO 2 — Configura el Entorno y la API

## El setup es igual en todos los proyectos — memoriza este patrón

### 2.1 Instalación de dependencias base

Estas librerías aparecen en casi todos los proyectos GenAI con LangChain:

```python
# El stack mínimo para proyectos GenAI con Claude/Gemini/OpenAI
pip install langchain                    # Framework de orquestación
pip install langchain-anthropic          # Conector para Claude (Anthropic)
pip install langchain-google-genai       # Conector para Gemini (Google)
pip install langchain-openai             # Conector para GPT (OpenAI)
pip install python-dotenv                # Gestión de API keys en archivo .env
pip install pandas numpy scikit-learn    # Stack ML clásico
```

### 2.2 Gestión de API Keys — NUNCA pongas claves en el código

```python
# Crea un archivo .env en la raíz del proyecto con:
# ANTHROPIC_API_KEY=sk-ant-...
# GOOGLE_API_KEY=AIza...
# OPENAI_API_KEY=sk-...

import os
from dotenv import load_dotenv

load_dotenv()  # Lee el archivo .env automáticamente

API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("API Key cargada:", "✅" if API_KEY else "❌ No encontrada")
```

### 2.3 Inicializar el LLM — El "motor" del pipeline

```python
# Con Claude (Anthropic) — RECOMENDADO para proyectos reales
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-sonnet-4-6",   # Modelo actual recomendado
    api_key=API_KEY,
    temperature=0.0,             # 0 = determinista (ideal para extracción/clasificación)
                                 # 0.7 = creativo (ideal para generación de texto)
    max_tokens=1024              # Limita el coste por llamada
)

# Con Gemini (Google) — usado en los notebooks del módulo
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
    temperature=0.3
)
```

### 2.4 ¿Qué temperatura usar?

| Temperatura | Cuándo usarla | Ejemplo |
|-------------|---------------|---------|
| `0.0` | Extracción de datos, clasificación, JSON | Parsear un ticket de soporte |
| `0.1 - 0.3` | Análisis, explicaciones | Explicar por qué un cliente hace churn |
| `0.5 - 0.7` | Generación creativa controlada | Escribir respuestas de soporte |
| `0.8 - 1.0` | Brainstorming, variedad máxima | Generar ideas de marketing |

# 📝 PASO 3 — Diseña el Prompt

## El prompt es el "algoritmo" en IA Generativa — diseñarlo bien lo es todo

Un prompt bien diseñado puede transformar completamente la calidad de los resultados. 
A diferencia del ML clásico donde optimizas hiperparámetros, aquí optimizas texto.

---

### 3.1 Anatomía de un prompt profesional

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
[ROL DEL SISTEMA]
Eres un {rol_descripcion}. Tu objetivo es {objetivo}.

[CONTEXTO]
{contexto_del_problema}

[TAREA]
Dado el siguiente input:
{input_variable}

[INSTRUCCIONES DE FORMATO]
Responde ÚNICAMENTE con este formato JSON (sin explicaciones, sin markdown):
{{
  "campo1": "valor",
  "campo2": número entre 0 y 1,
  "campo3": true o false
}}
""")
```

### 3.2 Las 4 partes clave de un prompt

| Parte | Propósito | Tip |
|-------|-----------|-----|
| **Rol** | Define la "identidad" del LLM | Sé específico: "analista financiero senior" es mejor que "experto" |
| **Contexto** | Da información de fondo | Incluye el contexto del usuario o del negocio |
| **Tarea** | Qué debe hacer exactamente | Usa verbos imperativos: "extrae", "clasifica", "genera" |
| **Formato** | Cómo debe responder | Siempre especifica el formato de salida para parsear después |

### 3.3 Cómo construir la Chain (cadena de procesamiento)

```python
from langchain_core.output_parsers import StrOutputParser

# El operador | encadena pasos: prompt → llm → parser
# Esto se llama LCEL (LangChain Expression Language)
chain = prompt | llm | StrOutputParser()

# Invocar la chain con variables
resultado = chain.invoke({
    "rol_descripcion": "analista de retención de clientes",
    "objetivo": "detectar clientes en riesgo de abandono",
    "contexto_del_problema": "empresa de SaaS con 10.000 clientes",
    "input_variable": "datos del cliente aquí"
})
```

### 3.4 Técnicas avanzadas de prompting

```python
# FEW-SHOT: dar ejemplos de input → output esperado
prompt_few_shot = ChatPromptTemplate.from_template("""
Clasifica el sentimiento del texto. Ejemplos:

Input: "Me encanta este producto"  → Output: positivo
Input: "Malísimo, una basura"      → Output: negativo  
Input: "Funciona, nada especial"   → Output: neutro

Ahora clasifica este texto:
Input: "{texto}"
Output:
""")

# CHAIN OF THOUGHT: pedirle que razone paso a paso
prompt_cot = ChatPromptTemplate.from_template("""
Analiza este caso de negocio paso a paso:
1. Primero identifica los hechos relevantes
2. Luego analiza las causas posibles
3. Finalmente da una recomendación

Caso: {caso}
""")
```

### 3.5 Reglas de oro del prompt engineering

```
✅ DO:
  - Sé específico sobre el formato de salida (JSON, lista, número)
  - Usa ejemplos cuando el formato es complejo (few-shot)
  - Indica restricciones: "máximo 15 palabras", "solo uno de: A|B|C"
  - Escapa las llaves en f-strings: {{ }} en lugar de { }

❌ DON'T:
  - Prompts vagos: "analiza esto" sin especificar qué analizar
  - Esperar JSON sin pedirlo explícitamente
  - Mezclar demasiadas tareas en un solo prompt
  - Olvidar manejar el caso en que el LLM no siga el formato
```

# 🗂️ PASO 4 — Prepara los Datos

## La entrada de datos en GenAI es diferente al ML clásico

En ML clásico preparas tablas de números. En GenAI, el dato puede ser texto,
y el LLM tiene que transformarlo. Este paso define cómo alimentas el pipeline.

---

### 4.1 Los 3 tipos de entrada en proyectos GenAI

```
TIPO A: Datos tabulares puros
  → Usas el LLM para GENERAR explicaciones de predicciones ML
  → Ejemplo: churn prediction + LLM que explica por qué
  → Notebook T4.1

TIPO B: Texto libre que necesita estructurarse
  → Usas el LLM para EXTRAER datos del texto → JSON → ML
  → Ejemplo: emails de soporte → clasificación automática
  → Notebooks T4.2, T4.3, T4.5

TIPO C: Preguntas en lenguaje natural sobre datos
  → Usas el LLM como INTERFAZ de consulta sobre un dataset
  → Ejemplo: "¿Qué región vendió más en Q3?"
  → Notebook T4.4
```

### 4.2 Creación de datasets sintéticos para prototipado

```python
import numpy as np
import pandas as pd

# Patrón estándar para crear datos de prueba rápidos
np.random.seed(42)  # Siempre fija la semilla para reproducibilidad
n = 500

df = pd.DataFrame({
    "feature_numerica": np.random.randint(1, 100, n),
    "feature_continua": np.random.uniform(0, 1000, n).round(2),
    "feature_categorica": np.random.choice(["A", "B", "C"], n),
    "texto_libre": ["descripción de ejemplo"] * n,
})

# Regla sintética para el target (hazla explícita y comprensible)
df["target"] = (
    (df["feature_numerica"] > 70) |
    (df["feature_continua"] < 100)
).astype(int)

print(f"Dataset: {df.shape}, Target rate: {df['target'].mean():.1%}")
```

### 4.3 Cómo pasar datos al LLM — Del DataFrame al Prompt

```python
# Patrón para iterar filas y llamar al LLM
resultados = []

for idx, fila in df.iterrows():
    resultado = chain.invoke({
        "campo1": fila["feature_numerica"],
        "campo2": fila["feature_categorica"],
        "campo3": fila["texto_libre"]
    })
    resultados.append(resultado)
    
    # Muestra progreso en datasets grandes
    if idx % 50 == 0:
        print(f"Procesado: {idx}/{len(df)}")

df["resultado_llm"] = resultados
```

### 4.4 Parsear la salida JSON del LLM — El paso crítico

```python
import json

def parsear_respuesta_llm(texto_crudo: str) -> dict:
    """
    El LLM a veces añade markdown alrededor del JSON.
    Esta función lo limpia y lo parsea de forma segura.
    """
    # Limpiar backticks de markdown que el LLM puede añadir
    limpio = texto_crudo.strip()
    limpio = limpio.replace("```json", "").replace("```", "").strip()
    
    try:
        return json.loads(limpio)
    except json.JSONDecodeError as e:
        print(f"⚠️ Error parseando JSON: {e}")
        print(f"Texto recibido: {texto_crudo[:200]}")
        return {}  # Retorna dict vacío para no romper el pipeline

# Uso
raw = chain.invoke({"input": "texto de prueba"})
datos = parsear_respuesta_llm(raw)
```

### 4.5 Consideraciones de coste y velocidad

```
Coste por llamada API (aproximado, 2025):
  - Claude Haiku:  $0.0001 por 1K tokens  (más barato, menos capaz)
  - Claude Sonnet: $0.003  por 1K tokens  (balance ideal)
  - Claude Opus:   $0.015  por 1K tokens  (más capaz, más caro)
  - Gemini Flash:  $0.0001 por 1K tokens  (muy económico)

Estrategia de coste:
  1. Prototipar con muestra pequeña (50-100 registros)
  2. Estimar coste total = tokens_promedio × n_registros × precio
  3. En producción, usar cascada (Paso 5.5) para reducir llamadas al LLM
```

# 🏗️ PASO 5 — Construye el Pipeline

## Los 6 patrones de pipeline — elige el que corresponde a tu problema

Esta es la parte central del proyecto. Dependiendo del Paso 1, usarás uno (o varios) 
de estos patrones. Cada uno está detallado en su notebook de ejemplo.

---

### PATRÓN A: ML predice → LLM explica  *(Notebook T4.1)*

```
Cuándo usarlo: Tienes un modelo ML ya entrenado y necesitas hacer sus predicciones 
comprensibles para usuarios no técnicos (negocio, clientes, reguladores).

Flujo:
  datos_cliente → modelo_ML → prediccion + probabilidad
                                         ↓
                              prompt con features + prediccion
                                         ↓
                                       LLM
                                         ↓
                             explicacion en lenguaje natural
```

```python
# Patrón de código
def predecir_y_explicar(datos_entrada: dict) -> dict:
    # Paso 1: predicción ML
    prediccion = modelo_ml.predict([lista_features])[0]
    probabilidad = modelo_ml.predict_proba([lista_features])[0][1]
    
    # Paso 2: explicación LLM
    explicacion = chain_explicacion.invoke({
        **datos_entrada,
        "prediccion": "CHURN" if prediccion == 1 else "OK",
        "probabilidad": probabilidad
    })
    
    return {"prediccion": prediccion, "probabilidad": probabilidad, "explicacion": explicacion}
```

---

### PATRÓN B: Texto → LLM extrae JSON → ML downstream  *(Notebook T4.2)*

```
Cuándo usarlo: Tienes texto no estructurado (emails, tickets, formularios) 
que necesitas convertir a datos tabulares para procesarlos con ML.

Flujo:
  texto_libre → prompt de extracción → LLM → JSON estructurado
                                                     ↓
                                           DataFrame / Base de datos
                                                     ↓
                                         modelo ML / dashboard / reglas
```

```python
# El LLM extrae campos específicos del texto
prompt_extraccion = """Extrae estos campos del texto en JSON:
{{"nombre": ..., "urgencia": "alta|media|baja", "sentimiento": -1.0 a 1.0}}
Texto: {texto}"""

datos_extraidos = json.loads(chain.invoke({"texto": texto_crudo}))
```

---

### PATRÓN C: Texto → LLM genera features → ML enriquecido  *(Notebook T4.3)*

```
Cuándo usarlo: Tienes datos tabulares + columnas de texto. 
El LLM genera features semánticas que el ML clásico no puede calcular.

Flujo:
  features_numericas ─────────────────────────────────┐
                                                       ↓
  texto_libre → LLM → score_sentimiento, intención... → concat → Modelo ML
                       menciona_precio, nivel_urgencia             mejorado
```

```python
# El LLM genera números que enriquecen el dataset
features_semanticas = {
    "score_sentimiento": -1.0 a 1.0,
    "intencion_recompra": 0.0 a 1.0,
    "menciona_devolucion": True/False,
    "nivel_frustracion": 0.0 a 1.0
}
df_enriquecido = pd.concat([df_original, pd.DataFrame(features_semanticas)], axis=1)
```

---

### PATRÓN D: LLM orquesta herramientas ML  *(Notebook T4.4)*

```
Cuándo usarlo: Quieres que usuarios no técnicos hagan análisis 
de datos con preguntas en lenguaje natural.

Flujo:
  pregunta_usuario → Agente LLM → decide qué herramientas usar
                                         ↓
                         tool_estadisticas() | tool_clustering() | tool_regresion()
                                         ↓
                              LLM sintetiza resultados en lenguaje natural
```

```python
@tool
def analizar_ventas_por_region(metrica: str) -> str:
    """Compara el rendimiento de las regiones para una métrica dada."""
    resultado = df.groupby("region")[metrica].mean().to_dict()
    return json.dumps(resultado)

agente = create_tool_calling_agent(llm, [analizar_ventas_por_region, ...], prompt)
ejecutor = AgentExecutor(agent=agente, tools=tools, verbose=True)
ejecutor.invoke({"input": "¿Qué región tiene mejor rendimiento?"})
```

---

### PATRÓN E: Cascada Reglas → ML → LLM  *(Notebook T4.5)*

```
Cuándo usarlo: Necesitas procesar MUCHOS textos con balance coste/precisión.
Los casos obvios los resuelven las capas baratas. El LLM solo ve casos ambiguos.

Flujo:
  texto → [CAPA 1: Reglas] → decide si confianza alta → resultado
                ↓ no decidido
          [CAPA 2: ML]     → decide si confianza > umbral → resultado
                ↓ no decidido
          [CAPA 3: LLM]    → siempre decide → resultado

Coste típico: LLM procesa solo 10-20% de los casos → 80-90% de ahorro
```

```python
def pipeline_cascada(texto: str):
    resultado = capa_reglas(texto)   # gratis, instantáneo
    if resultado: return resultado
    
    resultado = capa_ml(texto)       # barato, ~1ms
    if resultado: return resultado
    
    return capa_llm(texto)           # caro, ~2s — solo si las otras fallan
```

---

### PATRÓN F: LLM evalúa la calidad de otro LLM  *(Notebook T4.6)*

```
Cuándo usarlo: Necesitas comparar prompts, modelos o configuraciones 
sin anotadores humanos. Esencial para iterar y mejorar pipelines.

Flujo:
  pregunta + respuesta_referencia + respuesta_candidata
                          ↓
                    LLM-como-Juez
                          ↓
         {puntuacion: 8/10, fortalezas: [...], debilidades: [...]}
```

```python
prompt_juez = """Evalúa esta respuesta del 1 al 10.
Referencia: {referencia}
Respuesta a evaluar: {respuesta}
Responde en JSON: {{"puntuacion": ..., "razon": "..."}}"""

evaluacion = chain_juez.invoke({"referencia": ref, "respuesta": candidata})
```

# 📊 PASO 6 — Evalúa los Resultados

## En GenAI no basta con accuracy — necesitas evaluar calidad semántica

Este es el mayor cambio conceptual respecto al ML clásico. En clasificación 
tienes accuracy, F1, AUC. En GenAI la salida es texto libre, y necesitas 
métricas diferentes.

---

### 6.1 Métricas clásicas (cuando tienes ground truth)

```python
from sklearn.metrics import classification_report, accuracy_score

# Cuando el LLM clasifica (sí/no, categorías fijas)
# puedes usar las métricas de siempre
y_pred_llm = [resultado["decision"] for resultado in resultados]
y_true = [etiqueta_real for etiqueta_real in dataset["label"]]

print(classification_report(y_true, y_pred_llm))
```

### 6.2 LLM-as-Judge: cuando la salida es texto libre

```python
# El LLM juez evalúa la calidad de las respuestas de otro LLM
# Esto sustituye la anotación humana costosa

prompt_juez = ChatPromptTemplate.from_template("""
Evalúa esta respuesta del 1 al 10 según estos criterios: {criterios}

Respuesta de referencia: {referencia}
Respuesta a evaluar: {respuesta}

Responde en JSON:
{{"puntuacion_global": 8,
  "claridad": 9,
  "precision_tecnica": 7,
  "fortalezas": "...",
  "debilidades": "...",
  "justificacion": "..."}}
""")
```

### 6.3 Comparación A/B de prompts

```python
# Siempre compara al menos 2 variantes de prompt
scores_prompt_a = []  # prompt básico
scores_prompt_b = []  # prompt optimizado con contexto

for pregunta in benchmark:
    resp_a = chain_a.invoke({"pregunta": pregunta})
    resp_b = chain_b.invoke({"pregunta": pregunta, "contexto": contexto})
    
    eval_a = llm_juez.evaluate(resp_a)
    eval_b = llm_juez.evaluate(resp_b)
    
    scores_prompt_a.append(eval_a["puntuacion_global"])
    scores_prompt_b.append(eval_b["puntuacion_global"])

print(f"Prompt A: {np.mean(scores_prompt_a):.2f}/10")
print(f"Prompt B: {np.mean(scores_prompt_b):.2f}/10")
print(f"Mejora:   {np.mean(scores_prompt_b) - np.mean(scores_prompt_a):+.2f} pts")
```

### 6.4 Métricas de sistema (producción)

```python
# Más allá de la calidad, mide el comportamiento del sistema
metricas_sistema = {
    "latencia_p50_ms": ...,       # tiempo mediana de respuesta
    "latencia_p99_ms": ...,       # tiempo en el peor 1% de casos
    "tasa_error_json": ...,       # % de veces que el LLM no devuelve JSON válido
    "coste_por_registro": ...,    # USD por cada llamada al LLM
    "pct_resuelto_por_capa": {    # en pipelines en cascada
        "reglas": 0.45,
        "ml": 0.35,
        "llm": 0.20
    }
}
```

### 6.5 Tabla resumen: ¿Qué métrica usar en cada caso?

| Tipo de salida | Métrica recomendada | Cuándo usar |
|----------------|---------------------|-------------|
| Clasificación binaria | Accuracy, F1, AUC | Siempre que tengas labels |
| Texto libre con referencia | ROUGE, BERTScore | Resúmenes, traducciones |
| Texto libre sin referencia | LLM-as-Judge | Lo más frecuente |
| Rankings / ordenaciones | Correlación de Spearman | Comparar sistemas |
| Sistema en producción | Latencia, coste, error rate | Monitoreo continuo |

# 🔁 PASO 7 — Itera y Mejora

## El ciclo de mejora continua en proyectos GenAI

A diferencia del ML clásico donde entrenas una vez y deploys, en GenAI el ciclo 
de mejora es continuo y más rápido. Cambiar un prompt tarda minutos, no horas.

---

### 7.1 El ciclo de mejora del prompt

```
1. Ejecuta el pipeline con el prompt actual
2. Identifica los casos donde falla (LLM-as-Judge o revisión manual)
3. Analiza el patrón de fallos: ¿qué tienen en común los casos malos?
4. Modifica el prompt: añade instrucciones, ejemplos, o restricciones
5. Vuelve al paso 1 y mide si mejoró
```

### 7.2 ¿Cuándo cambiar de modelo?

```
Síntomas de que necesitas un modelo más potente:
  ❌ El LLM no sigue el formato JSON aunque lo pidas explícitamente
  ❌ Las respuestas son incoherentes o contradictorias
  ❌ El razonamiento es superficial en tareas complejas
  
Síntomas de que puedes usar un modelo más barato:
  ✅ La tarea es simple (clasificación binaria, extracción de campos fijos)
  ✅ Tienes many-shot examples que guían al modelo
  ✅ La latencia y el coste importan más que la calidad perfecta
```

### 7.3 Estrategias de mejora por orden de coste

```
BAJO COSTE (prueba primero):
  1. Mejorar el prompt (añadir contexto, ejemplos, formato más claro)
  2. Ajustar temperatura (baja para extracción, sube para creatividad)
  3. Añadir few-shot examples en el prompt

COSTE MEDIO:
  4. Cambiar a un modelo más potente
  5. Usar chain-of-thought (pedir que razone antes de responder)
  6. Dividir la tarea en múltiples pasos (chains encadenadas)

COSTE ALTO (solo si todo lo anterior falla):
  7. Fine-tuning del modelo con datos propios
  8. RAG (Retrieval-Augmented Generation) para añadir contexto
  9. Cambiar completamente la arquitectura del pipeline
```

### 7.4 Registro de experimentos — no improvises

```python
# Mantén un log de qué probaste y qué resultados obtuviste
experimentos = [
    {
        "version": "v1",
        "descripcion": "Prompt básico sin ejemplos",
        "modelo": "gemini-flash",
        "temperatura": 0.3,
        "score_llm_judge": 6.2,
        "tasa_error_json": 0.15,
        "notas": "Falla en casos ambiguos, formato JSON inconsistente"
    },
    {
        "version": "v2", 
        "descripcion": "Añadidos 3 ejemplos few-shot + formato JSON más explícito",
        "modelo": "gemini-flash",
        "temperatura": 0.0,
        "score_llm_judge": 7.8,
        "tasa_error_json": 0.02,
        "notas": "Mejora significativa. Temperatura 0 elimina variabilidad en JSON"
    },
]

# Compara experimentos
df_exp = pd.DataFrame(experimentos)
print(df_exp[["version", "score_llm_judge", "tasa_error_json"]].to_string())
```

# 🚀 PASO 8 — Empaqueta el Pipeline como Función Reutilizable

## El paso final antes de producción: encapsular todo en código limpio

Un proyecto GenAI bien hecho termina con funciones que cualquiera puede llamar
sin entender la implementación interna.

---

### 8.1 Patrón de función reutilizable

```python
def mi_pipeline_genai(entrada: dict, configuracion: dict = None) -> dict:
    """
    Documentación clara: qué recibe, qué devuelve, posibles excepciones.
    
    Args:
        entrada: diccionario con los datos de entrada
        configuracion: parámetros opcionales (modelo, temperatura, etc.)
    
    Returns:
        diccionario con: prediccion, confianza, explicacion, metadatos
    """
    config = configuracion or {}
    
    # 1. Validar entrada
    campos_requeridos = ["campo_a", "campo_b"]
    for campo in campos_requeridos:
        if campo not in entrada:
            raise ValueError(f"Campo requerido: {campo}")
    
    # 2. Ejecutar pipeline
    try:
        resultado_ml = modelo.predict(...)
        resultado_llm = chain.invoke(...)
        
        return {
            "prediccion": resultado_ml,
            "explicacion": resultado_llm,
            "modelo_version": "v2.1",
            "timestamp": datetime.now().isoformat()
        }
    
    except Exception as e:
        # Siempre maneja errores en producción
        return {"error": str(e), "entrada": entrada}
```

### 8.2 Checklist antes de considerar el proyecto terminado

```
✅ CÓDIGO
  □ Pipeline encapsulado en funciones con docstrings claros
  □ Manejo de errores cuando el LLM devuelve JSON inválido
  □ API keys en .env, nunca hardcodeadas
  □ Semilla aleatoria fijada (np.random.seed) para reproducibilidad

✅ EVALUACIÓN
  □ Métricas calculadas sobre un conjunto de test representativo
  □ Al menos 2 variantes de prompt comparadas (A/B)
  □ Casos de fallo analizados y documentados
  □ Coste por llamada estimado y aceptable

✅ DOCUMENTACIÓN (mínima)
  □ README con: qué hace, cómo instalarlo, cómo ejecutarlo
  □ Ejemplo de entrada/salida documentado en el notebook
  □ Limitaciones conocidas del sistema anotadas

✅ PRODUCCIÓN (si aplica)
  □ Rate limiting: no exceder cuota de la API
  □ Retry logic: reintentar si la API falla temporalmente
  □ Logging: registrar cada llamada y su resultado
  □ Monitoreo: alertas si la tasa de error sube
```

### 8.3 Arquitectura de referencia para proyectos completos

```
mi_proyecto_genai/
├── .env                    # API keys (nunca en git)
├── .env.example            # Plantilla de keys (sí en git)
├── requirements.txt        # Dependencias
├── notebooks/
│   ├── 01_exploracion.ipynb      # EDA y prototipado
│   ├── 02_pipeline.ipynb         # Desarrollo del pipeline
│   └── 03_evaluacion.ipynb       # Métricas y LLM-as-Judge
├── src/
│   ├── pipeline.py               # Funciones del pipeline
│   ├── prompts.py                # Todos los prompts centralizados
│   └── evaluacion.py             # Funciones de evaluación
└── tests/
    └── test_pipeline.py          # Tests unitarios
```

# 🗺️ RESUMEN — El Mapa del Proceso Completo

## De un vistazo: los 8 pasos de cualquier proyecto GenAI

```
┌─────────────────────────────────────────────────────────────────┐
│  PASO 1: Define el problema                                     │
│  → ¿Qué tipo de datos? ¿Qué salida quieres? ¿Restricciones?     │
│  → Elige el PATRÓN (A-F) que corresponde a tu caso              │
└────────────────────────┬────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────────┐
│  PASO 2: Configura el entorno                                   │
│  → Instala dependencias, carga API key, inicializa el LLM       │
│  → Elige temperatura según la tarea                             │
└────────────────────────┬────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────────┐
│  PASO 3: Diseña el prompt                                       │
│  → Rol + Contexto + Tarea + Formato de salida                   │
│  → Empieza simple, añade few-shot si falla                      │
└────────────────────────┬────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────────┐
│  PASO 4: Prepara los datos                                      │
│  → Crea o carga el dataset, define cómo pasa al LLM             │
│  → Implementa el parser JSON seguro                             │
└────────────────────────┬────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────────┐
│  PASO 5: Construye el pipeline                                  │
│  → Implementa el patrón elegido en el Paso 1                    │
│  → Chain LCEL: prompt | llm | parser                            │
└────────────────────────┬────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────────┐
│  PASO 6: Evalúa los resultados                                  │
│  → Métricas clásicas si tienes ground truth                     │
│  → LLM-as-Judge si la salida es texto libre                     │
└────────────────────────┬────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────────┐
│  PASO 7: Itera y mejora                                         │
│  → Analiza fallos → mejora prompt → mide → repite               │
│  → Cambia modelo solo si el prompt no es suficiente             │
└────────────────────────┬────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────────────┐
│  PASO 8: Empaqueta como función reutilizable                    │
│  → Encapsula, maneja errores, documenta, haz checklist          │
└─────────────────────────────────────────────────────────────────┘
```

---

## Decisión rápida: ¿Qué patrón necesito?

```
Mi dato de entrada es...

  TEXTO LIBRE → ¿Qué quiero hacer con él?
    ├── Extraer campos estructurados → PATRÓN B (T4.2)
    ├── Generar features para ML     → PATRÓN C (T4.3)  
    └── Clasificar/moderar           → PATRÓN E cascada (T4.5)

  DATOS TABULARES → ¿Qué quiero hacer?
    ├── Explicar predicciones ML     → PATRÓN A (T4.1)
    └── Consultar con lenguaje nat.  → PATRÓN D agente (T4.4)

  RESPUESTAS DE UN LLM → ¿Quiero evaluarlas?
    └── Evaluar calidad automát.     → PATRÓN F LLM-Judge (T4.6)
```

---

> **Recuerda:** Los notebooks T4.1 a T4.6 son ejemplos concretos de cada patrón.
> Esta guía es tu brújula. Los notebooks son los mapas detallados de cada destino.

# 📘 Vista Aérea de IA Generativa - Guía de Consulta Rápida


## 🎯 El Ecosistema Completo en una Imagen

```
┌────────────────────────────────────────────────────────────────────────┐
│                    IA GENERATIVA EN PROYECTOS REALES                   │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │           CAPA 1: FUNDAMENTOS (siempre presentes)               │   │
│  │  • LLM (GPT, Claude, Gemini)                                    │   │
│  │  • Prompts (Rol + Contexto + Tarea + Formato)                   │   │
│  │  • API keys, temperatura, tokens                                │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                              ↓                                         │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │        CAPA 2: PATRONES DE ARQUITECTURA (los 6 que viste)       │   │
│  │                                                                 │   │
│  │   A. ML + Explicación      D. Agente con Herramientas           │   │
│  │   B. Extracción            E. Cascada (optimización coste)      │   │
│  │   C. Feature Engineering   F. Evaluación (LLM-as-Judge)         │   │
│  │                                                                 │   │
│  │   → Son "recetas de cocina" para problemas típicos              │   │
│  │   → Puedes combinarlos (A+B, C+E, etc.)                         │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                              ↓                                         │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │      CAPA 3: PROCESO DE TRABAJO (los 8 pasos, siempre igual)    │   │
│  │                                                                 │   │
│  │   1. Define → 2. Configura → 3. Prompt → 4. Datos               │   │
│  │   5. Pipeline → 6. Evalúa → 7. Itera → 8. Empaqueta             │   │
│  │                                                                 │   │
│  │   → Es el "ritmo" de trabajo, independiente del patrón          │   │
│  │   → Algunos pasos son más relevantes según el patrón            │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```


---

## ❓ Pregunta Clave: ¿Son estos 6 los únicos patrones?

### Respuesta Corta
**No son los únicos que existen en el mundo, pero sí son los 6 patrones fundamentales que cubren el 80-90% de los casos de uso empresarial** con IA Generativa.

### ¿Por qué estos 6 y no otros?

| Patrón | Problema que resuelve | Frecuencia en industria |
|--------|----------------------|------------------------|
| A | Explicar predicciones ML | ⭐⭐⭐⭐⭐ Muy común |
| B | Estructurar texto caótico | ⭐⭐⭐⭐⭐ Muy común |
| C | Enriquecer ML con texto | ⭐⭐⭐⭐☆ Común |
| D | Interfaces conversacionales | ⭐⭐⭐⭐☆ Común |
| E | Escala con presupuesto limitado | ⭐⭐⭐⭐⭐ Muy común |
| F | Evaluar sin anotadores humanos | ⭐⭐⭐⭐☆ Común |

---

## 🔍 Otros Patrones que Existen (fuera de estos 6)

| Patrón adicional | Descripción | Cuándo aparece |
|------------------|-------------|----------------|
| **RAG (Retrieval Augmented Generation)** | LLM + búsqueda en documentos | Ya lo viste en PDF aparte (3_X_RAG) |
| **Fine-tuning** | Entrenar el LLM con datos propios | Cuando necesitas especialización profunda |
| **Multi-agente** | Varios agentes colaborando | Sistemas complejos (fuera de scope del máster) |
| **Chain-of-Thought** | LLM razonando paso a paso | Dentro del prompting, no es arquitectura |

> **Nota:** El RAG ya lo estudiaste en el PDF "3_X_RAG_Retrieval_Augmented_Generation". Es el patrón 7, pero se enseña aparte porque es más complejo y tiene su propio módulo.

---

## ✅ La Verdad Simplificada

```
┌─────────────────────────────────────────────────────────┐
│  EN LA PRÁCTICA REAL, CASI TODO SE RESUELVE CON:        │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  1. Un LLM + un buen prompt (base de todo)              │
│                                                         │
│  2. Uno de estos 6 patrones (según tu problema):        │
│     • ¿ML existente? → A (explicar)                     │
│     • ¿Texto sin estructura? → B (extraer)              │
│     • ¿Datos mixtos? → C (features)                     │
│     • ¿Chat sobre datos? → D (agente)                   │
│     • ¿Mucho volumen? → E (cascada)                     │
│     • ¿Evaluar calidad? → F (juez)                      │
│     • ¿Documentos externos? → RAG (el 7º)               │
│                                                         │
│  3. Los 8 pasos de proceso (siempre los mismos)         │
│                                                         │
└─────────────────────────────────────────────────────────┘
```


---

## 🧭 Brújula de Decisión Rápida

```
¿CUÁL ES TU PREGUNTA CLAVE?

├─ ¿Tengo un modelo ML y quiero explicarlo? ───────→ PATRÓN A
│
├─ ¿Tengo texto desordenado y quiero organizarlo? ─→ PATRÓN B
│
├─ ¿Tengo texto + números y quiero mejorar predicciones? ─→ PATRÓN C
│
├─ ¿Quiero que la gente pregunte en lenguaje natural? ───→ PATRÓN D
│
├─ ¿Proceso mucho contenido y el dinero importa? ──→ PATRÓN E
│
├─ ¿Necesito evaluar si mis respuestas son buenas? ─→ PATRÓN F
│
└─ ¿Quiero que el LLM lea mis documentos? ─────────→ RAG (patrón 7)
```


---

## 📊 Tabla de Decisión Rápida

| Si tu problema es... | Usa el patrón... | Entrada | Salida |
|---------------------|------------------|---------|--------|
| Explicar predicciones de ML | **A** | Datos tabulares + predicción | Texto explicativo |
| Convertir texto a datos | **B** | Texto libre | JSON estructurado |
| Mejorar ML con texto | **C** | Texto + datos numéricos | Features + predicción mejorada |
| Interfaz conversacional | **D** | Pregunta en lenguaje natural | Respuesta + acciones |
| Procesar mucho con poco dinero | **E** | Texto | Decisión (sí/no) optimizada |
| Evaluar calidad automáticamente | **F** | Respuestas LLM | Puntuación 1-10 |
| Responder desde documentos | **RAG** | Pregunta + base documental | Respuesta fundamentada |

---

## 🔗 Combinaciones Comunes de Patrones

| Combinación | Descripción | Caso de uso |
|-------------|-------------|-------------|
| **A + B** | Extraer datos → Explicar predicción | Email → Datos → ML predice → LLM explica |
| **C + E** | Features semánticas + Cascada | Moderación inteligente con contexto |
| **B + D** | Extraer datos → Agente conversacional | Chatbot que entiende emails y responde |
| **D + F** | Agente + Evaluación continua | Mejorar el chatbot midiendo sus respuestas |

---

## 💡 Recordatorio Clave

> **"No hay una estrategia única que funcione mejor en todos los casos; la elección depende de las particularidades del caso de uso. Muchas veces, las técnicas pueden combinarse para obtener mejores resultados."**
>
> — Guía metodológica 001_Proceso_IA_Generativa

**Estos 6 patrones (+ RAG) son los más comunes y cubren la gran mayoría de proyectos.** No son los únicos que existen teóricamente, pero son los que realmente se usan en la industria. El máster te enseña estos porque son **prácticos, reutilizables y componibles**.

---

## 📚 Referencias Rápidas

- **Guía general:** `001_Proceso_IA_Generativa_Claude.ipynb` (8 pasos del proceso)
- **Patrones A-F:** `T4.1` a `T4.6` (implementaciones prácticas)
- **Patrón RAG:** `3_X_RAG_Retrieval_Augmented_Generation_Claude.ipynb`
- **Diccionarios:** `diccionario_1_L_Introduccion_LLMs.md` y `diccionario_3_X_RAG.md`
